# Create histone features

In [1]:
# Import Library
import polars as pl
import os
import numpy as np
import torch

## Define Required Functions

### Define Gene Expression schema

In [2]:
# Define the gene expression schema
schema = pl.Schema({
        'chromosome_name': pl.String,
        'start': pl.Int64,
        'end': pl.Int64,
        'gene_id': pl.String,
        'E066': pl.Float64,
        'strand': pl.Int64,
        'label': pl.Int64,
        'external_gene_name': pl.String,
        'start_position': pl.Int64,
        'end_position': pl.Int64,
        'tss': pl.Int64
})

In [3]:
DATASET_PATH = '../dataset/E066/'

### Define the function for getting histone features

In [4]:
# def get_histone_features(genes_df, histone_df, histone_name):
#     schema = pl.Schema({
#         'gene_id': pl.String,
#         histone_name: pl.List(pl.Float64),
#         f'{histone_name}_wc': pl.UInt32,
#         f'{histone_name}_len': pl.UInt32
#     })

#     final_result = pl.DataFrame(schema=schema)

#     for i, chunk in enumerate(genes_df.iter_slices(n_rows=100)):
    
#         # Create a sequence of window starts and window ends for each gene
#         genes_with_windows = genes_df.with_columns([
#                 pl.int_ranges(pl.col('start'), pl.col('end'), 100).alias('window_starts')
#             ]).explode('window_starts')
    
#         genes_with_windows = genes_with_windows.with_columns([
#                 (pl.col('window_starts') + 100).alias('window_ends')
#             ])
    
        
    
#         # Join genes with histone data
#         joined = genes_with_windows.join(
#             histone_df,
#             left_on='gene_id',
#             right_on='gene_id',
#             how='left'
#         )
    
#         # Filter and calculate average signal value
#         result = joined.filter(
#             (pl.col('chromStart') <= pl.col('window_starts')) & 
#             (pl.col('chromEnd') >= pl.col('window_ends')) |
#             (pl.col('chromStart') >= pl.col('window_starts')) & 
#             (pl.col('chromStart') <= pl.col('window_ends')) & 
#             (pl.col('chromEnd') >= pl.col('window_ends')) |
#             (pl.col('chromStart') <= pl.col('window_starts')) & 
#             (pl.col('chromEnd') >= pl.col('window_starts')) & 
#             (pl.col('chromEnd') <= pl.col('window_ends'))
#         ).group_by(['gene_id', 'window_starts'], maintain_order=True).agg([
#             pl.col('signalValue').mean().alias(histone_name)
#         ]).sort(['gene_id', 'window_starts'])
    
#         # Find the gene without histone match
#         gene_wo_histone = genes_with_windows.filter(~pl.col('window_starts').is_in(result['window_starts']))
    
#         # Restructured the dataframe so it can be merged with results
#         gene_wo_histone = gene_wo_histone.with_columns(
#             chromosome_name_right = pl.lit(None).cast(pl.String),
#             start_right = pl.lit(0).cast(pl.Int64),
#             end_right =  pl.lit(0).cast(pl.Int64),
#             E066_right = pl.lit(0.0).cast(pl.Float64),
#             strand_right = pl.lit(0).cast(pl.Int64),
#             label_right = pl.lit(0).cast(pl.Int64),
#             external_gene_name_right = pl.lit(None).cast(pl.String),
#             start_position_right = pl.lit(0).cast(pl.Int64),
#             end_position_right = pl.lit(0).cast(pl.Int64),
#             tss_right = pl.lit(0).cast(pl.Int64),
#             chrom = pl.lit(None).cast(pl.String),
#             chromStart = pl.lit(0).cast(pl.Int64),
#             chromEnd = pl.lit(0).cast(pl.Int64),
#             name = pl.lit(None).cast(pl.String),
#             score = pl.lit(0).cast(pl.Int64),
#             strand_peak = pl.lit(None).cast(pl.String),
#             signalValue = pl.lit(0.0).cast(pl.Float64),
#             pValue = pl.lit(0.0).cast(pl.Float64),
#             qValue = pl.lit(0.0).cast(pl.Float64),
#             peak = pl.lit(0).cast(pl.Int64),
#             startBucket = pl.lit(0.0).cast(pl.Float64),
#             endBucket = pl.lit(0.0).cast(pl.Float64)
#         )
    
#         # Grouping so it can be merged with result dataframe
#         gene_wo_histone = gene_wo_histone.group_by(['gene_id', 'window_starts'], maintain_order=True).agg([
#             pl.col('signalValue').mean().alias(histone_name)
#         ]).sort(['gene_id', 'window_starts'])
    
#         # Merging with the result dataframe
#         result.extend(gene_wo_histone)
    
#         # Sorting by window_start
#         result = result.sort("window_starts")
    
#         # Final group by
#         result = result.group_by(['gene_id'], maintain_order=True).agg(
#             pl.col(histone_name)
#         ).sort('gene_id')
    
#         result = result.with_columns(
#             pl.col(histone_name)
#             .list.eval(pl.element().is_not_null() & (pl.element() > 0))
#             .list.sum()
#             .alias(f"{histone_name}_wc")
#         )
    
#         result = result.with_columns(
#             pl.col(histone_name).list.len().alias(f"{histone_name}_len")
#         )

#         final_result.extend(result)

#     return final_result

In [5]:
def get_histone_features(genes_df, histone_df, histone_name):
    
    # Create a sequence of window starts and window ends for each gene
    genes_with_windows = genes_df.with_columns([
            pl.int_ranges(pl.col('start'), pl.col('end'), 100).alias('window_starts')
        ]).explode('window_starts')

    genes_with_windows = genes_with_windows.with_columns([
            (pl.col('window_starts') + 100).alias('window_ends')
        ])

    # Join genes with histone data
    joined = genes_with_windows.join(
        histone_df,
        left_on='gene_id',
        right_on='gene_id',
        how='left'
    )

    # Filter and calculate average signal value
    result = joined.filter(
        (pl.col('chromStart') <= pl.col('window_starts')) & 
        (pl.col('chromEnd') >= pl.col('window_ends')) |
        (pl.col('chromStart') >= pl.col('window_starts')) & 
        (pl.col('chromStart') <= pl.col('window_ends')) & 
        (pl.col('chromEnd') >= pl.col('window_ends')) |
        (pl.col('chromStart') <= pl.col('window_starts')) & 
        (pl.col('chromEnd') >= pl.col('window_starts')) & 
        (pl.col('chromEnd') <= pl.col('window_ends'))
    ).group_by(['gene_id', 'window_starts'], maintain_order=True).agg([
        pl.col('signalValue').mean().alias(histone_name)
    ]).sort(['gene_id', 'window_starts'])

    # Find the gene without histone match
    gene_wo_histone = genes_with_windows.filter(~pl.col('window_starts').is_in(result['window_starts']))

    # Restructured the dataframe so it can be merged with results
    gene_wo_histone = gene_wo_histone.with_columns(
        chromosome_name_right = pl.lit(None).cast(pl.String),
        start_right = pl.lit(0).cast(pl.Int64),
        end_right =  pl.lit(0).cast(pl.Int64),
        E066_right = pl.lit(0.0).cast(pl.Float64),
        strand_right = pl.lit(0).cast(pl.Int64),
        label_right = pl.lit(0).cast(pl.Int64),
        external_gene_name_right = pl.lit(None).cast(pl.String),
        start_position_right = pl.lit(0).cast(pl.Int64),
        end_position_right = pl.lit(0).cast(pl.Int64),
        tss_right = pl.lit(0).cast(pl.Int64),
        chrom = pl.lit(None).cast(pl.String),
        chromStart = pl.lit(0).cast(pl.Int64),
        chromEnd = pl.lit(0).cast(pl.Int64),
        name = pl.lit(None).cast(pl.String),
        score = pl.lit(0).cast(pl.Int64),
        strand_peak = pl.lit(None).cast(pl.String),
        signalValue = pl.lit(0.0).cast(pl.Float64),
        pValue = pl.lit(0.0).cast(pl.Float64),
        qValue = pl.lit(0.0).cast(pl.Float64),
        peak = pl.lit(0).cast(pl.Int64),
        startBucket = pl.lit(0.0).cast(pl.Float64),
        endBucket = pl.lit(0.0).cast(pl.Float64)
    )

    # Grouping so it can be merged with result dataframe
    gene_wo_histone = gene_wo_histone.group_by(['gene_id', 'window_starts'], maintain_order=True).agg([
        pl.col('signalValue').mean().alias(histone_name)
    ]).sort(['gene_id', 'window_starts'])

    # Merging with the result dataframe
    result.extend(gene_wo_histone)

    # Sorting by window_start
    result = result.sort("window_starts")

    # Final group by
    result = result.group_by(['gene_id'], maintain_order=True).agg(
        pl.col(histone_name)
    ).sort('gene_id')

    result = result.with_columns(
        pl.col(histone_name)
        .list.eval(pl.element().is_not_null() & (pl.element() > 0))
        .list.sum()
        .alias(f"{histone_name}_wc")
    )

    result = result.with_columns(
        pl.col(histone_name).list.len().alias(f"{histone_name}_len")
    )

    return result

### Function for getting empty histone dataframe

In [6]:
def create_empty_dataframe(histone_name):
    
    schema = pl.Schema({
        'gene_id': pl.String,
        histone_name: pl.List(pl.Float64),
        f'{histone_name}_wc': pl.UInt32,
        f'{histone_name}_len': pl.UInt32
    })

    df = pl.DataFrame(schema=schema)

    return df

### Function for getting histone in chunk

In [7]:
# Getting histone in chunk
def get_histone_features_chunk(genes, histones, histone_name):
    
    gene_w_histone = create_empty_dataframe(histone_name)
    
    for i, chunk in enumerate(genes.iter_slices(n_rows=100)):
        result = get_histone_features(chunk, histones, histone_name)
        gene_w_histone.extend(result)

    return gene_w_histone

## Loading the dataset

In [8]:
# Read gene expression file with chromosome and +/- 5k from TSS
E066_pl = pl.read_csv(os.path.join(DATASET_PATH, "E066.bed"), 
                      separator="\t", 
                      schema=schema,
                      has_header=False,
                      skip_rows=0)

In [9]:
E066_pl

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988
"""chrX""",99834799,99844799,"""ENSG00000000005""",0.191,1,-1,"""TNMD""",99839799,99854882,99839799
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092
"""chr1""",169858408,169868408,"""ENSG00000000457""",4.733,-1,1,"""SCYL3""",169818772,169863408,169863408
"""chr1""",169626245,169636245,"""ENSG00000000460""",0.942,1,-1,"""C1orf112""",169631245,169823221,169631245
…,…,…,…,…,…,…,…,…,…,…
"""chr15""",102280913,102290913,"""ENSG00000259658""",0.212,-1,-1,"""RP11-89K11.1""",102277302,102285913,102285913
"""chr15""",97966182,97976182,"""ENSG00000259664""",0.0,-1,-1,"""CTD-2147F2.2""",97913601,97971182,97971182
"""chr16""",33642696,33652696,"""ENSG00000259680""",0.071,-1,-1,"""RP11-812E19.9""",33647044,33647696,33647696


In [10]:
# Loading histone file
H3K4me1_pl = pl.read_csv(os.path.join(DATASET_PATH, "E066_H3K4me1_df.csv"))
H3K4me3_pl = pl.read_csv(os.path.join(DATASET_PATH, "E066_H3K4me3_df.csv"))
H3K9me3_pl = pl.read_csv(os.path.join(DATASET_PATH, "E066_H3K9me3_df.csv"))
H3K27me3_pl = pl.read_csv(os.path.join(DATASET_PATH, "E066_H3K27me3_df.csv"))
H3K36me3_pl = pl.read_csv(os.path.join(DATASET_PATH, "E066_H3K36me3_df.csv"))

## Find the gene and histone intersection

In [11]:
gene_w_H3K4me1 = get_histone_features_chunk(E066_pl, H3K4me1_pl, 'H3K4me1')
gene_w_H3K4me3 = get_histone_features_chunk(E066_pl, H3K4me3_pl, 'H3K4me3')
gene_w_H3K9me3 = get_histone_features_chunk(E066_pl, H3K9me3_pl, 'H3K9me3')
gene_w_H3K27me3 = get_histone_features_chunk(E066_pl, H3K27me3_pl, 'H3K27me3')
gene_w_H3K36me3 = get_histone_features_chunk(E066_pl, H3K36me3_pl, 'H3K36me3')

In [12]:
gene_w_H3K4me1

gene_id,H3K4me1,H3K4me1_wc,H3K4me1_len
str,list[f64],u32,u32
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",9,100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",52,100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",35,100
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100
…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100


In [13]:
gene_w_H3K4me3

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len
str,list[f64],u32,u32
"""ENSG00000000003""","[0.0, 4.19024, … 0.0]",19,100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",36,100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",35,100
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100
…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100


In [14]:
gene_w_H3K9me3

gene_id,H3K9me3,H3K9me3_wc,H3K9me3_len
str,list[f64],u32,u32
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",2,100
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",2,100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000460""","[3.0104, 3.0104, … 0.0]",5,100
…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100


In [15]:
gene_w_H3K27me3

gene_id,H3K27me3,H3K27me3_wc,H3K27me3_len
str,list[f64],u32,u32
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100
…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",5,100
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100


In [16]:
gene_w_H3K36me3

gene_id,H3K36me3,H3K36me3_wc,H3K36me3_len
str,list[f64],u32,u32
"""ENSG00000000003""","[3.39243, 0.0, … 0.0]",1,100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000419""","[10.55422, 10.55422, … 0.0]",25,100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100
…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100


## Join all histones into single dataframe

In [17]:
# Join all histone into single dataframe
gene_w_histone = gene_w_H3K4me1 \
                    .join(gene_w_H3K4me3, on='gene_id') \
                    .join(gene_w_H3K9me3, on='gene_id') \
                    .join(gene_w_H3K27me3, on='gene_id') \
                    .join(gene_w_H3K36me3, on='gene_id')

In [18]:
gene_w_histone

gene_id,H3K4me1,H3K4me1_wc,H3K4me1_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K27me3,H3K27me3_wc,H3K27me3_len,H3K36me3,H3K36me3_wc,H3K36me3_len
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",9,100,"[0.0, 4.19024, … 0.0]",19,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[3.39243, 0.0, … 0.0]",1,100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",2,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",52,100,"[0.0, 0.0, … 0.0]",36,100,"[0.0, 0.0, … 0.0]",2,100,"[0.0, 0.0, … 0.0]",0,100,"[10.55422, 10.55422, … 0.0]",25,100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[3.0104, 3.0104, … 0.0]",5,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",5,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100


## Join with E066 label

In [19]:
# Load E066 dataset
E066_w_label = pl.read_csv(os.path.join(DATASET_PATH, 'E066_merged.csv'))

In [20]:
E066_w_label = E066_w_label.select(['gene_id', 'E066', 'label'])

In [21]:
E066_w_label

gene_id,E066,label
str,f64,i64
"""ENSG00000000003""",73.205,1
"""ENSG00000000005""",0.191,-1
"""ENSG00000000419""",52.609,1
"""ENSG00000000457""",4.733,1
"""ENSG00000000460""",0.942,-1
…,…,…
"""ENSG00000259658""",0.212,-1
"""ENSG00000259664""",0.0,-1
"""ENSG00000259680""",0.071,-1


In [22]:
# Join with gene_w_histone
gene_w_histone = gene_w_histone.join(
    E066_w_label,
    on = 'gene_id'
)

In [23]:
gene_w_histone

gene_id,H3K4me1,H3K4me1_wc,H3K4me1_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K27me3,H3K27me3_wc,H3K27me3_len,H3K36me3,H3K36me3_wc,H3K36me3_len,E066,label
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64,i64
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",9,100,"[0.0, 4.19024, … 0.0]",19,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[3.39243, 0.0, … 0.0]",1,100,73.205,1
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",2,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.191,-1
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",52,100,"[0.0, 0.0, … 0.0]",36,100,"[0.0, 0.0, … 0.0]",2,100,"[0.0, 0.0, … 0.0]",0,100,"[10.55422, 10.55422, … 0.0]",25,100,52.609,1
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,4.733,1
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[3.0104, 3.0104, … 0.0]",5,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.942,-1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.212,-1
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",5,100,"[0.0, 0.0, … 0.0]",0,100,0.0,-1
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.071,-1


## Concat histone columns into single column

In [25]:
gene_w_histone = gene_w_histone.with_columns(
    pl.struct(['H3K4me1', 'H3K4me3', 'H3K9me3', 'H3K27me3', 'H3K36me3']).map_elements(
        lambda x: [x['H3K4me1'], x['H3K4me3'], x['H3K9me3'], x['H3K27me3'], x['H3K36me3']],
        return_dtype = pl.List(pl.List(pl.Float64))
    )
    .alias('all_histone')
)

In [26]:
gene_w_histone

gene_id,H3K4me1,H3K4me1_wc,H3K4me1_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K27me3,H3K27me3_wc,H3K27me3_len,H3K36me3,H3K36me3_wc,H3K36me3_len,E066,label,all_histone
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64,i64,list[list[f64]]
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",9,100,"[0.0, 4.19024, … 0.0]",19,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[3.39243, 0.0, … 0.0]",1,100,73.205,1,"[[0.0, 0.0, … 0.0], [0.0, 4.19024, … 0.0], … [3.39243, 0.0, … 0.0]]"
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",2,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.191,-1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",52,100,"[0.0, 0.0, … 0.0]",36,100,"[0.0, 0.0, … 0.0]",2,100,"[0.0, 0.0, … 0.0]",0,100,"[10.55422, 10.55422, … 0.0]",25,100,52.609,1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [10.55422, 10.55422, … 0.0]]"
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,4.733,1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[3.0104, 3.0104, … 0.0]",5,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.942,-1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.212,-1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",5,100,"[0.0, 0.0, … 0.0]",0,100,0.0,-1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.071,-1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"


## Save into parquet file

In [27]:
# Save to parquet
gene_w_histone.write_parquet(os.path.join(DATASET_PATH, 'E066_w_histone_pl.parquet'))

## Read from parquet

In [28]:
E066_w_histone_pl = pl.read_parquet(os.path.join(DATASET_PATH, 'E066_w_histone_pl.parquet'))

In [29]:
E066_w_histone_pl

gene_id,H3K4me1,H3K4me1_wc,H3K4me1_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K27me3,H3K27me3_wc,H3K27me3_len,H3K36me3,H3K36me3_wc,H3K36me3_len,E066,label,all_histone
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64,i64,list[list[f64]]
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",9,100,"[0.0, 4.19024, … 0.0]",19,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[3.39243, 0.0, … 0.0]",1,100,73.205,1,"[[0.0, 0.0, … 0.0], [0.0, 4.19024, … 0.0], … [3.39243, 0.0, … 0.0]]"
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",2,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.191,-1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",52,100,"[0.0, 0.0, … 0.0]",36,100,"[0.0, 0.0, … 0.0]",2,100,"[0.0, 0.0, … 0.0]",0,100,"[10.55422, 10.55422, … 0.0]",25,100,52.609,1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [10.55422, 10.55422, … 0.0]]"
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,4.733,1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[3.0104, 3.0104, … 0.0]",5,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.942,-1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.212,-1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",5,100,"[0.0, 0.0, … 0.0]",0,100,0.0,-1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.071,-1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"


# Polars 'join' inconsistent results

Test with small dataset

## Using manual step

In [30]:
# gene_sample = E066_pl.filter(pl.col('gene_id').is_in(['ENSG00000259133', 'ENSG00000016082']))
gene_sample = E066_pl.limit(200)

In [31]:
gene_sample

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988
"""chrX""",99834799,99844799,"""ENSG00000000005""",0.191,1,-1,"""TNMD""",99839799,99854882,99839799
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092
"""chr1""",169858408,169868408,"""ENSG00000000457""",4.733,-1,1,"""SCYL3""",169818772,169863408,169863408
"""chr1""",169626245,169636245,"""ENSG00000000460""",0.942,1,-1,"""C1orf112""",169631245,169823221,169631245
…,…,…,…,…,…,…,…,…,…,…
"""chr6""",16124356,16134356,"""ENSG00000007944""",4.253,1,1,"""MYLIP""",16129356,16148479,16129356
"""chrX""",100124334,100134334,"""ENSG00000007952""",0.0,-1,-1,"""NOX1""",100098313,100129334,100129334
"""chr1""",23852712,23862712,"""ENSG00000007968""",0.103,-1,-1,"""E2F2""",23832922,23857712,23857712


In [32]:
gene_sample = gene_sample.with_columns([
        pl.int_ranges(pl.col('start'), pl.col('end'), 100).alias('window_starts')
    ]).explode('window_starts')

In [33]:
gene_sample = gene_sample.with_columns([
        (pl.col('window_starts') + 100).alias('window_ends')
    ])

In [34]:
gene_sample

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss,window_starts,window_ends
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64,i64,i64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99889988,99890088
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890088,99890188
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890188,99890288
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890288,99890388
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890388,99890488
…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrX""",47474252,47484252,"""ENSG00000008056""",0.602,-1,-1,"""SYN1""",47431303,47479252,47479252,47483752,47483852
"""chrX""",47474252,47484252,"""ENSG00000008056""",0.602,-1,-1,"""SYN1""",47431303,47479252,47479252,47483852,47483952
"""chrX""",47474252,47484252,"""ENSG00000008056""",0.602,-1,-1,"""SYN1""",47431303,47479252,47479252,47483952,47484052


In [35]:
# Join genes with histone data
joined = gene_sample.join(
    H3K4me1_pl,
    left_on='gene_id',
    right_on='gene_id',
    how='left'
)

In [36]:
joined

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss,window_starts,window_ends,chromosome_name_right,start_right,end_right,E066_right,strand_right,label_right,external_gene_name_right,start_position_right,end_position_right,tss_right,chrom,chromStart,chromEnd,name,score,strand_peak,signalValue,pValue,qValue,peak,startBucket,endBucket
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64,i64,i64,str,i64,i64,f64,i64,i64,str,i64,i64,i64,str,i64,i64,str,i64,str,f64,f64,f64,i64,f64,f64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99889988,99890088,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890088,99890188,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890188,99890288,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890288,99890388,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890388,99890488,"""chrX""",99889988,99899988,73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrX""",47474252,47484252,"""ENSG00000008056""",0.602,-1,-1,"""SYN1""",47431303,47479252,47479252,47483752,47483852,"""chrX""",47474252,47484252,0.602,-1,-1,"""SYN1""",47431303,47479252,47479252,"""chrX""",47478956,47479213,"""Rank_131100""",62,""".""",3.76658,6.20494,4.46705,121,47.04,49.61
"""chrX""",47474252,47484252,"""ENSG00000008056""",0.602,-1,-1,"""SYN1""",47431303,47479252,47479252,47483852,47483952,"""chrX""",47474252,47484252,0.602,-1,-1,"""SYN1""",47431303,47479252,47479252,"""chrX""",47478956,47479213,"""Rank_131100""",62,""".""",3.76658,6.20494,4.46705,121,47.04,49.61
"""chrX""",47474252,47484252,"""ENSG00000008056""",0.602,-1,-1,"""SYN1""",47431303,47479252,47479252,47483952,47484052,"""chrX""",47474252,47484252,0.602,-1,-1,"""SYN1""",47431303,47479252,47479252,"""chrX""",47478956,47479213,"""Rank_131100""",62,""".""",3.76658,6.20494,4.46705,121,47.04,49.61


In [37]:
# Filter and calculate average signal value
result = joined.filter(
    (pl.col('chromStart') <= pl.col('window_starts')) & 
    (pl.col('chromEnd') >= pl.col('window_ends')) |
    (pl.col('chromStart') >= pl.col('window_starts')) & 
    (pl.col('chromStart') <= pl.col('window_ends')) & 
    (pl.col('chromEnd') >= pl.col('window_ends')) |
    (pl.col('chromStart') <= pl.col('window_starts')) & 
    (pl.col('chromEnd') >= pl.col('window_starts')) & 
    (pl.col('chromEnd') <= pl.col('window_ends'))
).group_by(['gene_id', 'window_starts'], maintain_order=True).agg([
    pl.col('signalValue').mean().alias('H3K4me1')
]).sort(['gene_id', 'window_starts'])

In [38]:
result

gene_id,window_starts,H3K4me1
str,i64,f64
"""ENSG00000000003""",99890188,5.92171
"""ENSG00000000003""",99890288,5.92171
"""ENSG00000000003""",99890388,5.92171
"""ENSG00000000003""",99890488,5.92171
"""ENSG00000000003""",99890588,5.92171
…,…,…
"""ENSG00000008018""",170865329,13.81079
"""ENSG00000008018""",170865429,13.81079
"""ENSG00000008056""",47478952,3.76658


In [39]:
# Find the gene without histone match
gene_wo_histone = gene_sample.filter(~pl.col('window_starts').is_in(result['window_starts']))

In [40]:
gene_wo_histone

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss,window_starts,window_ends
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64,i64,i64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99889988,99890088
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890088,99890188
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99891088,99891188
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99891188,99891288
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99891288,99891388
…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrX""",47474252,47484252,"""ENSG00000008056""",0.602,-1,-1,"""SYN1""",47431303,47479252,47479252,47483752,47483852
"""chrX""",47474252,47484252,"""ENSG00000008056""",0.602,-1,-1,"""SYN1""",47431303,47479252,47479252,47483852,47483952
"""chrX""",47474252,47484252,"""ENSG00000008056""",0.602,-1,-1,"""SYN1""",47431303,47479252,47479252,47483952,47484052


In [41]:
# Restructured the dataframe so it can be merged with results
gene_wo_histone = gene_wo_histone.with_columns(
    chromosome_name_right = pl.lit(None).cast(pl.String),
    start_right = pl.lit(0).cast(pl.Int64),
    end_right =  pl.lit(0).cast(pl.Int64),
    E066_right = pl.lit(0.0).cast(pl.Float64),
    strand_right = pl.lit(0).cast(pl.Int64),
    label_right = pl.lit(0).cast(pl.Int64),
    external_gene_name_right = pl.lit(None).cast(pl.String),
    start_position_right = pl.lit(0).cast(pl.Int64),
    end_position_right = pl.lit(0).cast(pl.Int64),
    tss_right = pl.lit(0).cast(pl.Int64),
    chrom = pl.lit(None).cast(pl.String),
    chromStart = pl.lit(0).cast(pl.Int64),
    chromEnd = pl.lit(0).cast(pl.Int64),
    name = pl.lit(None).cast(pl.String),
    score = pl.lit(0).cast(pl.Int64),
    strand_peak = pl.lit(None).cast(pl.String),
    signalValue = pl.lit(0.0).cast(pl.Float64),
    pValue = pl.lit(0.0).cast(pl.Float64),
    qValue = pl.lit(0.0).cast(pl.Float64),
    peak = pl.lit(0).cast(pl.Int64),
    startBucket = pl.lit(0.0).cast(pl.Float64),
    endBucket = pl.lit(0.0).cast(pl.Float64)
)

In [42]:
gene_wo_histone

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss,window_starts,window_ends,chromosome_name_right,start_right,end_right,E066_right,strand_right,label_right,external_gene_name_right,start_position_right,end_position_right,tss_right,chrom,chromStart,chromEnd,name,score,strand_peak,signalValue,pValue,qValue,peak,startBucket,endBucket
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64,i64,i64,str,i64,i64,f64,i64,i64,str,i64,i64,i64,str,i64,i64,str,i64,str,f64,f64,f64,i64,f64,f64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99889988,99890088,null,0,0,0.0,0,0,null,0,0,0,null,0,0,null,0,null,0.0,0.0,0.0,0,0.0,0.0
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99890088,99890188,null,0,0,0.0,0,0,null,0,0,0,null,0,0,null,0,null,0.0,0.0,0.0,0,0.0,0.0
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99891088,99891188,null,0,0,0.0,0,0,null,0,0,0,null,0,0,null,0,null,0.0,0.0,0.0,0,0.0,0.0
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99891188,99891288,null,0,0,0.0,0,0,null,0,0,0,null,0,0,null,0,null,0.0,0.0,0.0,0,0.0,0.0
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,99891288,99891388,null,0,0,0.0,0,0,null,0,0,0,null,0,0,null,0,null,0.0,0.0,0.0,0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrX""",47474252,47484252,"""ENSG00000008056""",0.602,-1,-1,"""SYN1""",47431303,47479252,47479252,47483752,47483852,null,0,0,0.0,0,0,null,0,0,0,null,0,0,null,0,null,0.0,0.0,0.0,0,0.0,0.0
"""chrX""",47474252,47484252,"""ENSG00000008056""",0.602,-1,-1,"""SYN1""",47431303,47479252,47479252,47483852,47483952,null,0,0,0.0,0,0,null,0,0,0,null,0,0,null,0,null,0.0,0.0,0.0,0,0.0,0.0
"""chrX""",47474252,47484252,"""ENSG00000008056""",0.602,-1,-1,"""SYN1""",47431303,47479252,47479252,47483952,47484052,null,0,0,0.0,0,0,null,0,0,0,null,0,0,null,0,null,0.0,0.0,0.0,0,0.0,0.0


In [43]:
# Grouping so it can be merged with result dataframe
gene_wo_histone = gene_wo_histone.group_by(['gene_id', 'window_starts']).agg([
    pl.col('signalValue').mean().alias('H3K4me1')
]).sort(['gene_id', 'window_starts'])

In [44]:
gene_wo_histone

gene_id,window_starts,H3K4me1
str,i64,f64
"""ENSG00000000003""",99889988,0.0
"""ENSG00000000003""",99890088,0.0
"""ENSG00000000003""",99891088,0.0
"""ENSG00000000003""",99891188,0.0
"""ENSG00000000003""",99891288,0.0
…,…,…
"""ENSG00000008056""",47483752,0.0
"""ENSG00000008056""",47483852,0.0
"""ENSG00000008056""",47483952,0.0


In [45]:
# Merging with the result dataframe
result.extend(gene_wo_histone)

# Sorting by window_start
result = result.sort("window_starts")

In [46]:
result

gene_id,window_starts,H3K4me1
str,i64,f64
"""ENSG00000007384""",121354,3.81793
"""ENSG00000007384""",121454,3.81793
"""ENSG00000007384""",121554,3.81793
"""ENSG00000007384""",121654,3.81793
"""ENSG00000007384""",121754,3.81793
…,…,…
"""ENSG00000006607""",242300158,3.01517
"""ENSG00000006607""",242300258,3.01517
"""ENSG00000006607""",242300358,3.01517


In [47]:
# Final group by
result = result.group_by(['gene_id'], maintain_order=True).agg(
    pl.col('H3K4me1')
).sort('gene_id')

In [48]:
result

gene_id,H3K4me1
str,list[f64]
"""ENSG00000000003""","[0.0, 0.0, … 0.0]"
"""ENSG00000000005""","[0.0, 0.0, … 0.0]"
"""ENSG00000000419""","[0.0, 0.0, … 0.0]"
"""ENSG00000000457""","[0.0, 0.0, … 0.0]"
"""ENSG00000000460""","[0.0, 0.0, … 0.0]"
…,…
"""ENSG00000007944""","[0.0, 0.0, … 0.0]"
"""ENSG00000007952""","[0.0, 0.0, … 0.0]"
"""ENSG00000007968""","[0.0, 0.0, … 0.0]"


In [49]:
result = result.with_columns(
    pl.col('H3K4me1')
    .list.eval(pl.element().is_not_null() & (pl.element() > 0))
    .list.sum()
    .alias("H3K4me1_wc")
)

In [50]:
result = result.with_columns(
    pl.col('H3K4me1').list.len().alias("H3K4me1_len")
)

In [51]:
result

gene_id,H3K4me1,H3K4me1_wc,H3K4me1_len
str,list[f64],u32,u32
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",9,100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",52,100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",35,100
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100
…,…,…,…
"""ENSG00000007944""","[0.0, 0.0, … 0.0]",26,100
"""ENSG00000007952""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000007968""","[0.0, 0.0, … 0.0]",19,100


In [52]:
result.filter(pl.col('H3K4me1_len') < 100)

gene_id,H3K4me1,H3K4me1_wc,H3K4me1_len
str,list[f64],u32,u32


## Using function

### Manual selection

In [53]:
# gene_sample2 = E066_pl.filter(pl.col('gene_id').is_in(['ENSG00000259133', 'ENSG00000016082']))
gene_sample2 = E066_pl.limit(300)
# gene_sample2 = E066_pl.gather_every(1000, offset=0)

In [54]:
gene_sample2

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988
"""chrX""",99834799,99844799,"""ENSG00000000005""",0.191,1,-1,"""TNMD""",99839799,99854882,99839799
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092
"""chr1""",169858408,169868408,"""ENSG00000000457""",4.733,-1,1,"""SCYL3""",169818772,169863408,169863408
"""chr1""",169626245,169636245,"""ENSG00000000460""",0.942,1,-1,"""C1orf112""",169631245,169823221,169631245
…,…,…,…,…,…,…,…,…,…,…
"""chr11""",17224530,17234530,"""ENSG00000011405""",6.67,-1,1,"""PIK3C2A""",17099277,17229530,17229530
"""chr19""",44169699,44179699,"""ENSG00000011422""",11.947,-1,1,"""PLAUR""",44150247,44174699,44174699
"""chr7""",36424415,36434415,"""ENSG00000011426""",0.451,1,-1,"""ANLN""",36429415,36493400,36429415


In [55]:
result2 = get_histone_features(gene_sample2, H3K4me1_pl, 'H3K4me1')

In [56]:
result2.schema

Schema([('gene_id', String),
        ('H3K4me1', List(Float64)),
        ('H3K4me1_wc', UInt32),
        ('H3K4me1_len', UInt32)])

In [57]:
result2

gene_id,H3K4me1,H3K4me1_wc,H3K4me1_len
str,list[f64],u32,u32
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",9,100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",52,100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",35,100
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100
…,…,…,…
"""ENSG00000011405""","[0.0, 0.0, … 0.0]",27,100
"""ENSG00000011422""","[0.0, 0.0, … 0.0]",14,100
"""ENSG00000011426""","[0.0, 0.0, … 0.0]",11,100


In [58]:
len(result2.filter(pl.col('H3K4me1_len') < 100))

0

In [59]:
result2.schema

Schema([('gene_id', String),
        ('H3K4me1', List(Float64)),
        ('H3K4me1_wc', UInt32),
        ('H3K4me1_len', UInt32)])

### Iter Slice

#### H3K4me1

In [60]:
histone_name = 'H3K4me1'
H3K4me1_schema = pl.Schema({
    'gene_id': pl.String,
    histone_name: pl.List(pl.Float64),
    f'{histone_name}_wc': pl.UInt32,
    f'{histone_name}_len': pl.UInt32
})

In [61]:
gene_w_H3K4me1 = pl.DataFrame(schema=H3K4me1_schema)

In [62]:
gene_w_H3K4me1

gene_id,H3K4me1,H3K4me1_wc,H3K4me1_len
str,list[f64],u32,u32


In [67]:
error_rows = 0

gene_w_H3K4me1 = pl.DataFrame(schema=H3K4me1_schema)

for i, chunk in enumerate(E066_pl.iter_slices(n_rows=20_000)):
    # print(f"Chunk {i+1}: , Chunk size: {len(chunk)}")
    result = get_histone_features(chunk, H3K4me1_pl, 'H3K4me1')
    non_100 = len(result.filter(pl.col('H3K4me1_len') < 100))
    if non_100 > 0:
        print(f"[ERROR] Chunk {i+1}: {non_100}")
        error_rows += non_100
    else:
        print(f"Chunk {i+1}: {non_100}")
    # print(result.head(5))
    # print("---")

    gene_w_H3K4me1.extend(result)

print("---")
print(f"Error rows: {error_rows}")

[ERROR] Chunk 1: 272
---
Error rows: 272


In [65]:
gene_w_H3K4me1.filter(pl.col('H3K4me1_len') < 100)

gene_id,H3K4me1,H3K4me1_wc,H3K4me1_len
str,list[f64],u32,u32
"""ENSG00000002746""","[0.0, 0.0, … 0.0]",21,64
"""ENSG00000005884""","[0.0, 0.0, … 0.0]",45,78
"""ENSG00000006625""","[0.0, 0.0, … 0.0]",12,79
"""ENSG00000006704""","[0.0, 0.0, … 0.0]",21,56
"""ENSG00000007952""","[0.0, 0.0, … 0.0]",0,88
…,…,…,…
"""ENSG00000253293""","[0.0, 0.0, … 3.87293]",26,96
"""ENSG00000255472""","[0.0, 0.0, … 5.35955]",5,69
"""ENSG00000256892""","[0.0, 0.0, … 0.0]",0,57
